# Pretraining Evaluation Notebook

This notebook evaluates a pretrained Swin3D classifier on RSNA multi-label injury classification.
All stages provide numeric outputs and visualizations.


In [ ]:
# Stage 0: Interactive Parameters
from pathlib import Path

REPO_DIR = Path.cwd().resolve()
CHECKPOINT_PATH = str(REPO_DIR / "checkpoints/pretrain/pretrain_best.pth")
DATA_DIR = str(REPO_DIR / "data/rsna2023")
METRICS_LOG_PATH = str(REPO_DIR / "logs/pretrain/latest/metrics.jsonl")
SPLIT = "val"  # "val" or "test"
DECISION_THRESHOLD = 0.5
SAVE_DIR = str(REPO_DIR / "eval_results")

# Optional: second checkpoint for Stage 7 comparison
CHECKPOINT_PATH_2 = None

assert SPLIT in {"val", "test"}, "SPLIT must be 'val' or 'test'"


In [ ]:
# Shared imports and constants
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from tqdm.auto import tqdm

from datasets.pretrain_dataset import RSNAPretrainDataset, pretrain_collate_fn, LABEL_COLUMNS
from models.swin3d_classifier import build_pretrain_model
from pretrain import load_config

ORGAN_GROUPS = {
    "bowel": [0, 1],
    "extravasation": [2, 3],
    "kidney": [4, 5, 6],
    "liver": [7, 8, 9],
    "spleen": [10, 11, 12],
}

SAVE_DIR = Path(SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


## Stage 1: Data Loading

Load dataset, create deterministic validation/test subsets, and display dataset statistics with label distribution.


In [ ]:
def create_eval_split(dataset, split_name: str, seed: int = 42):
    """Create a deterministic 80/10/10 split and return val or test subset.

    Args:
        dataset: Full RSNA pretraining dataset.
        split_name: Either "val" or "test".
        seed: Random seed used for reproducible shuffling.
    """
    n = len(dataset)
    indices = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    train_end = int(0.8 * n)
    val_end = int(0.9 * n)
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]

    if split_name == "val":
        return Subset(dataset, val_indices), val_indices
    return Subset(dataset, test_indices), test_indices


dataset = RSNAPretrainDataset(
    data_dir=DATA_DIR,
    csv_file="train_2024.csv",
    image_dir="train_images",
    augment=False,
)

eval_dataset, eval_indices = create_eval_split(dataset, SPLIT)

all_labels = np.stack([sample["labels"] for sample in dataset.samples], axis=0)
eval_labels = np.stack([dataset.samples[i]["labels"] for i in eval_indices], axis=0)

stats_df = pd.DataFrame([
    {"metric": "Total samples", "value": len(dataset)},
    {"metric": f"{SPLIT} samples", "value": len(eval_dataset)},
    {"metric": "Number of labels", "value": len(LABEL_COLUMNS)},
    {"metric": "Mean positive rate (full dataset)", "value": float(all_labels.mean())},
    {"metric": f"Mean positive rate ({SPLIT})", "value": float(eval_labels.mean())},
])
display(stats_df)

label_dist_df = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "positive_count": eval_labels.sum(axis=0).astype(int),
    "negative_count": (len(eval_labels) - eval_labels.sum(axis=0)).astype(int),
    "positive_rate": eval_labels.mean(axis=0),
})
display(label_dist_df)

plt.figure(figsize=(14, 5))
plt.bar(label_dist_df["label"], label_dist_df["positive_rate"], color="#3b82f6")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Positive Rate")
plt.xlabel("Label")
plt.title(f"Label Distribution on {SPLIT} Split")
plt.tight_layout()
plt.show()


## Stage 2: Model Loading

Load checkpoint and print model/checkpoint statistics.


In [ ]:
def load_model_from_checkpoint(checkpoint_path: str, device: torch.device):
    config = load_config("configs/pretrain_config.yaml")
    model = build_pretrain_model(config).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    model_info = {
        "total_parameters": int(sum(p.numel() for p in model.parameters())),
        "trainable_parameters": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "checkpoint_epoch": int(checkpoint.get("epoch", -1)),
        "best_val_auroc": float(checkpoint.get("best_val_auroc", float("nan"))),
        "checkpoint_keys": sorted(list(checkpoint.keys())) if isinstance(checkpoint, dict) else [],
    }
    return model, checkpoint, model_info


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, checkpoint, model_info = load_model_from_checkpoint(CHECKPOINT_PATH, device)

model_info_df = pd.DataFrame([
    {"field": "Device", "value": str(device)},
    {"field": "Total parameters", "value": model_info["total_parameters"]},
    {"field": "Trainable parameters", "value": model_info["trainable_parameters"]},
    {"field": "Checkpoint epoch", "value": model_info["checkpoint_epoch"]},
    {"field": "Best validation AUROC", "value": model_info["best_val_auroc"]},
])
display(model_info_df)
print("Checkpoint fields:", model_info["checkpoint_keys"])


## Stage 3: Batch Inference

Run inference on the selected split with a progress bar.


In [ ]:
@torch.no_grad()
def run_inference(model, eval_dataset, batch_size=2):
    """Run batched model inference and return logits, probabilities, and predictions.

    Args:
        model: Loaded Swin3D classifier.
        eval_dataset: Evaluation subset.
        batch_size: Mini-batch size (small default to control 3D memory usage).

    Returns:
        Dictionary with patient IDs, logits, labels, probabilities, and thresholded predictions.
    """
    loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=pretrain_collate_fn,
    )

    logits_list = []
    labels_list = []
    patient_ids = []

    for batch in tqdm(loader, desc=f"Inference on {SPLIT}"):
        volumes = batch["volumes"].to(device).float()
        labels = batch["labels"].cpu().numpy()

        outputs = model(volumes)
        logits = outputs["logits"].cpu().numpy()

        logits_list.append(logits)
        labels_list.append(labels)
        patient_ids.extend(batch["patient_ids"])

    logits = np.concatenate(logits_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    probs = torch.sigmoid(torch.from_numpy(logits)).numpy()
    preds = (probs >= DECISION_THRESHOLD).astype(np.int32)

    return {
        "patient_ids": patient_ids,
        "logits": logits,
        "labels": labels,
        "probs": probs,
        "preds": preds,
    }


inference_outputs = run_inference(model, eval_dataset)
print("Inference completed.")
print("Logits shape:", inference_outputs["logits"].shape)
print("Labels shape:", inference_outputs["labels"].shape)


## Stage 4: Metrics Calculation

Compute label-wise, organ-wise, and global metrics. Also parse training history from `metrics.jsonl`.


In [ ]:
def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_ap(y_true, y_prob):
    if np.sum(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_prob)


def compute_labelwise_metrics(labels, probs, preds):
    rows = []
    for idx, label_name in enumerate(LABEL_COLUMNS):
        y_true = labels[:, idx]
        y_prob = probs[:, idx]
        y_pred = preds[:, idx]
        rows.append({
            "label": label_name,
            "AUC": safe_auc(y_true, y_prob),
            "AP": safe_ap(y_true, y_prob),
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1-Score": f1_score(y_true, y_pred, zero_division=0),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows)


def compute_organ_metrics(label_df):
    rows = []
    for organ, indices in ORGAN_GROUPS.items():
        subset = label_df.iloc[indices]
        rows.append({
            "organ": organ,
            "Average AUC": subset["AUC"].mean(skipna=True),
            "Average AP": subset["AP"].mean(skipna=True),
            "Composite F1": subset["F1-Score"].mean(skipna=True),
        })
    return pd.DataFrame(rows)


def compute_global_summary(label_df):
    return {
        "mean_auc": float(label_df["AUC"].mean(skipna=True)),
        "mean_ap": float(label_df["AP"].mean(skipna=True)),
        "mean_accuracy": float(label_df["Accuracy"].mean(skipna=True)),
        "macro_f1": float(label_df["F1-Score"].mean(skipna=True)),
        "macro_precision": float(label_df["Precision"].mean(skipna=True)),
        "macro_recall": float(label_df["Recall"].mean(skipna=True)),
    }


def load_metrics_history(metrics_path: str):
    path = Path(metrics_path)
    if not path.exists():
        print(f"Warning: metrics log not found at {path}")
        return pd.DataFrame()

    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    history_df = pd.DataFrame(rows)
    if not history_df.empty:
        history_df = history_df.sort_values(["epoch", "phase"]).reset_index(drop=True)
    return history_df


labels = inference_outputs["labels"]
probs = inference_outputs["probs"]
preds = inference_outputs["preds"]

label_metrics_df = compute_labelwise_metrics(labels, probs, preds)
organ_metrics_df = compute_organ_metrics(label_metrics_df)
global_metrics = compute_global_summary(label_metrics_df)
history_df = load_metrics_history(METRICS_LOG_PATH)

display(label_metrics_df)
display(organ_metrics_df)
display(pd.DataFrame([global_metrics]))

print("Loaded history rows:", len(history_df))
if not history_df.empty:
    display(history_df.head())


## Stage 5: Visualization

Required plots:
1. Train vs validation loss curves
2. Mean AUROC (14-label average) per epoch
3. Organ-level mean AUROC curves per epoch
4. mAP per epoch (when available)


In [ ]:
def build_epoch_phase_table(history_df):
    """Pivot history into epoch x phase columns for plotting train/val curves."""
    if history_df.empty:
        return pd.DataFrame()
    return history_df.pivot_table(index="epoch", columns="phase", aggfunc="first")


def extract_organ_auroc_history(history_df, organ_metrics_df):
    """Build per-epoch organ AUROC history using available log columns.

    Priority order:
    1) direct organ AUROC columns (e.g., auroc_bowel)
    2) mean of per-label AUROC columns for each organ
    3) fallback to final evaluation organ AUC as a constant reference line
    """
    if history_df.empty:
        return pd.DataFrame()

    val_df = history_df[history_df["phase"] == "val"].copy()
    val_df = val_df.sort_values("epoch")
    organ_hist = pd.DataFrame({"epoch": val_df["epoch"].to_numpy()})

    for organ, indices in ORGAN_GROUPS.items():
        direct_col = f"auroc_{organ}"
        if direct_col in val_df.columns:
            organ_hist[organ] = val_df[direct_col].to_numpy()
            continue

        label_cols = []
        for idx in indices:
            key = f"auroc_{LABEL_COLUMNS[idx]}"
            if key in val_df.columns:
                label_cols.append(key)

        if label_cols:
            organ_hist[organ] = val_df[label_cols].mean(axis=1).to_numpy()
        else:
            fallback = organ_metrics_df.loc[organ_metrics_df["organ"] == organ, "Average AUC"].iloc[0]
            organ_hist[organ] = np.repeat(fallback, len(val_df))

    return organ_hist


epoch_phase_table = build_epoch_phase_table(history_df)

# Plot 1: train vs val loss curves
plt.figure(figsize=(8, 5))
if not epoch_phase_table.empty and ("loss_total", "train") in epoch_phase_table.columns:
    plt.plot(epoch_phase_table.index, epoch_phase_table[("loss_total", "train")], marker="o", label="train_loss")
if not epoch_phase_table.empty and ("loss_total", "val") in epoch_phase_table.columns:
    plt.plot(epoch_phase_table.index, epoch_phase_table[("loss_total", "val")], marker="o", label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss Across Epochs")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "loss_curves.png", dpi=200)
plt.show()

# Plot 2: average AUROC across 14 labels per epoch
plt.figure(figsize=(8, 5))
if not epoch_phase_table.empty and ("mean_auroc", "val") in epoch_phase_table.columns:
    plt.plot(epoch_phase_table.index, epoch_phase_table[("mean_auroc", "val")], marker="o", color="#2563eb")
plt.xlabel("Epoch")
plt.ylabel("Mean AUROC")
plt.title("Average AUROC (14 Labels) per Epoch")
plt.tight_layout()
plt.savefig(SAVE_DIR / "auroc_curves.png", dpi=200)
plt.show()

# Plot 3: per-organ AUROC curves
organ_hist_df = extract_organ_auroc_history(history_df, organ_metrics_df)
plt.figure(figsize=(9, 5))
if not organ_hist_df.empty:
    for organ in ORGAN_GROUPS.keys():
        plt.plot(organ_hist_df["epoch"], organ_hist_df[organ], marker="o", label=organ)
plt.xlabel("Epoch")
plt.ylabel("Organ AUROC")
plt.title("Average AUROC per Organ Across Epochs")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "organ_auroc_curves.png", dpi=200)
plt.show()

# Plot 4: mAP per epoch (from mean_ap when available)
plt.figure(figsize=(8, 5))
if not epoch_phase_table.empty and ("mean_ap", "val") in epoch_phase_table.columns:
    plt.plot(epoch_phase_table.index, epoch_phase_table[("mean_ap", "val")], marker="o", color="#16a34a")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("mAP per Epoch")
plt.tight_layout()
plt.savefig(SAVE_DIR / "map_curve.png", dpi=200)
plt.show()


## Stage 6: Error Analysis

Identify worst labels, compare organs, and summarize failure modes.


In [ ]:
worst_labels_df = label_metrics_df.sort_values("AUC", ascending=True).head(5).reset_index(drop=True)
worst_labels_df.insert(0, "rank", np.arange(1, len(worst_labels_df) + 1))

organ_compare_df = organ_metrics_df.sort_values("Average AUC", ascending=True).reset_index(drop=True)

failure_rows = []
for idx, label_name in enumerate(LABEL_COLUMNS):
    y_true = labels[:, idx]
    y_pred = preds[:, idx]
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    dominant_failure = "false_negative" if fn >= fp else "false_positive"
    failure_rows.append({
        "label": label_name,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "true_negative": tn,
        "dominant_failure_mode": dominant_failure,
    })

failure_modes_df = pd.DataFrame(failure_rows)

display(worst_labels_df)
display(organ_compare_df)
display(failure_modes_df.sort_values(["false_negative", "false_positive"], ascending=False).head(10))


## Stage 7: Comparison Analysis (Optional)

Compare current checkpoint with a second checkpoint when `CHECKPOINT_PATH_2` is provided.


In [ ]:
comparison_results = None

if CHECKPOINT_PATH_2:
    model_2, _, _ = load_model_from_checkpoint(CHECKPOINT_PATH_2, device)
    outputs_2 = run_inference(model_2, eval_dataset)

    label_metrics_df_2 = compute_labelwise_metrics(
        outputs_2["labels"], outputs_2["probs"], outputs_2["preds"]
    )
    global_metrics_2 = compute_global_summary(label_metrics_df_2)

    compare_df = label_metrics_df[["label", "AUC", "AP", "F1-Score"]].merge(
        label_metrics_df_2[["label", "AUC", "AP", "F1-Score"]],
        on="label",
        suffixes=("_ckpt1", "_ckpt2"),
    )
    compare_df["AUC_delta"] = compare_df["AUC_ckpt2"] - compare_df["AUC_ckpt1"]
    compare_df["AP_delta"] = compare_df["AP_ckpt2"] - compare_df["AP_ckpt1"]
    compare_df["F1_delta"] = compare_df["F1-Score_ckpt2"] - compare_df["F1-Score_ckpt1"]

    comparison_results = {
        "global_ckpt1": global_metrics,
        "global_ckpt2": global_metrics_2,
    }

    display(compare_df.sort_values("AUC_delta"))
    display(pd.DataFrame(comparison_results))
else:
    print("Comparison skipped. Set CHECKPOINT_PATH_2 to enable Stage 7.")


## Stage 8: Report Export

Export metrics and summary artifacts.


In [ ]:
report = {
    "checkpoint_path": CHECKPOINT_PATH,
    "data_dir": DATA_DIR,
    "split": SPLIT,
    "decision_threshold": DECISION_THRESHOLD,
    "global_metrics": global_metrics,
    "label_wise_metrics": label_metrics_df.to_dict(orient="records"),
    "organ_wise_metrics": organ_metrics_df.to_dict(orient="records"),
    "worst_labels": worst_labels_df.to_dict(orient="records"),
    "comparison": comparison_results,
    "output_files": {
        "loss_curves": str(SAVE_DIR / "loss_curves.png"),
        "auroc_curves": str(SAVE_DIR / "auroc_curves.png"),
        "organ_auroc_curves": str(SAVE_DIR / "organ_auroc_curves.png"),
        "map_curve": str(SAVE_DIR / "map_curve.png"),
    },
}

metrics_report_path = SAVE_DIR / "metrics_report.json"
with metrics_report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

summary_path = SAVE_DIR / "evaluation_summary.txt"
with summary_path.open("w", encoding="utf-8") as f:
    f.write("Pretraining Evaluation Summary
")
    f.write("=" * 40 + "
")
    f.write(f"Checkpoint: {CHECKPOINT_PATH}
")
    f.write(f"Data directory: {DATA_DIR}
")
    f.write(f"Split: {SPLIT}
")
    f.write(f"Decision threshold: {DECISION_THRESHOLD}

")

    f.write("Global Metrics
")
    for key, value in global_metrics.items():
        f.write(f"- {key}: {value:.6f}
")

    f.write("
Worst 5 Labels by AUC
")
    for _, row in worst_labels_df.iterrows():
        f.write(f"- {row['label']}: AUC={row['AUC']:.6f}, AP={row['AP']:.6f}, F1={row['F1-Score']:.6f}
")

    f.write("
Organ Metrics
")
    for _, row in organ_metrics_df.iterrows():
        f.write(
            f"- {row['organ']}: Average AUC={row['Average AUC']:.6f}, "
            f"Average AP={row['Average AP']:.6f}, Composite F1={row['Composite F1']:.6f}
"
        )

print("Saved:")
print("-", metrics_report_path)
print("-", SAVE_DIR / "loss_curves.png")
print("-", SAVE_DIR / "auroc_curves.png")
print("-", SAVE_DIR / "organ_auroc_curves.png")
print("-", SAVE_DIR / "map_curve.png")
print("-", summary_path)
